In [12]:
import torch
import numpy as np
import cv2
import os
from PIL import Image
from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
import functions

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

model_path = r".\data\models\Nanonets-OCR2-1.5B-exp"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    device_map="cuda",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model = model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_path)
processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)


def ocr_page_with_nanonets_s(input_data, model, processor, max_new_tokens=2048):
    prompt = """Extract the text from the above document as if you were reading it naturally. Return the tables in html format. Return the equations in LaTeX representation. If there is an image in the document and image caption is not present, add a small description of the image inside the <img></img> tag; otherwise, add the image caption inside <img></img>. Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number> or <page_number>9/22</page_number>. Prefer using ☐ and ☑ for check boxes."""

    image = None

    try:
        if isinstance(input_data, str):
            image = Image.open(input_data).convert("RGB")
        elif isinstance(input_data, np.ndarray):
            rgb_image = cv2.cvtColor(input_data, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(rgb_image)
        elif isinstance(input_data, Image.Image):
            image = input_data.convert("RGB")
        else:
            raise TypeError(f"Unsupported input type: {type(input_data)}")
    except Exception as e:
        print(f"Ошибка конвертации файла: {e}")

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]

    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)

    return output_text[0]


image_path = "data/docs/сканирование0001 (2).pdf"
save_path = image_path.replace("docs", "output").replace(".png", "_processed.png").replace(".pdf", "_processed.png")

file = functions.preprocess_for_ocr(image_path)
pil_image = Image.fromarray(cv2.cvtColor(file[0], cv2.COLOR_BGR2RGB))
result = ocr_page_with_nanonets_s(pil_image, model, processor)
print(result)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 446.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 14.25 GiB is allocated by PyTorch, and 140.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [9]:
import easyocr
import functions

reader = easyocr.Reader(["ru", "en"], gpu=True)
image_path = "data/docs/2126869_big.СНИЛС Эгамбердиев.id-o_1cpg8f5dtqeq1vf51bi71epncbc10.jpeg"
file = functions.preprocess_for_ocr(image_path)
results = reader.readtext(file[0], detail=1)

for (bbox, text, prob) in results:
    # bbox — это координаты четырех углов рамки текста
    # text — сам текст
    # prob — уверенность модели (0.0 - 1.0)
    print(f"Текст: {text} | Уверенность: {prob:.2f}")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Текст: Российская Федерация | Уверенность: 1.00
Текст: СТРАХОВОЕ СВИДЕТЕЛЬСТВО | Уверенность: 1.00
Текст: ОБЯЗАТЕЛЬНОГО ПЕНСИОННОГО СТРАХОВАНИЯ | Уверенность: 0.94
Текст: 172-204-837 43 | Уверенность: 0.76
Текст: Фн; 0 | Уверенность: 0.77
Текст: ЭГАМБЕРДИЕВ | Уверенность: 1.00
Текст: ИЛХОМДЖОН | Уверенность: 1.00
Текст: ДЖАХОНГИРОВИЧ | Уверенность: 1.00
Текст: Дала | Уверенность: 0.49
Текст: Ktrmo   powdkтпшя | Уверенность: 0.07
Текст: 27 октября 1983 | Уверенность: 0.97
Текст: УЙЧИНСКИЙ | Уверенность: 1.00
Текст: НАМАНГАНСКАЯ ОБЛАСТЬ | Уверенность: 1.00
Текст: МУЖСКОЙ | Уверенность: 1.00
Текст: Пол | Уверенность: 0.46
Текст: 14 мюня 2012 | Уверенность: 0.75
Текст: Даша  ренапрп | Уверенность: 0.24
